# PE10: Text Classification Model Comparison

Comparing Logistic Regression, Naive Bayes, LinearSVC, Decision Tree, and Random Forest classifiers on the 20 Newsgroups dataset using TF-IDF vectorization and macro-averaged evaluation metrics.

In [0]:
%pip install pandas numpy nltk scikit-learn

import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Download required NLTK data
nltk.download(['punkt', 'punkt_tab', 'stopwords', 'wordnet'], quiet=True)

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


True

In [0]:
# Part 1: Data Preprocessing

# Preprocessing function
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """Convert text to lowercase, tokenize, remove non-alphabetic tokens,
    remove stopwords, lemmatize, and join back into a string."""
    try:
        text = text.lower()
        tokens = word_tokenize(text)
        tokens = [t for t in tokens if t.isalpha()]
        tokens = [t for t in tokens if t not in stop_words]
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
        return ' '.join(tokens)
    except Exception:
        return ''

# Load dataset
categories = ['alt.atheism', 'comp.graphics', 'sci.med', 'soc.religion.christian']
print("Loading 20 Newsgroups dataset...")
train_data = fetch_20newsgroups(subset='train', categories=categories, remove=('headers', 'footers', 'quotes'))
test_data = fetch_20newsgroups(subset='test', categories=categories, remove=('headers', 'footers', 'quotes'))

# Combine train and test for our own split
all_texts = train_data.data + test_data.data
all_labels = list(train_data.target) + list(test_data.target)
label_names = train_data.target_names

print(f"Total documents: {len(all_texts)}")
print(f"Categories: {label_names}")

# Preprocess all texts
print("Preprocessing texts...")
processed_texts = [preprocess_text(text) for text in all_texts]

# Vectorize using TF-IDF
print("Vectorizing with TF-IDF...")
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)
X = vectorizer.fit_transform(processed_texts)
y = np.array(all_labels)

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

Loading 20 Newsgroups dataset...
Total documents: 3759
Categories: ['alt.atheism', 'comp.graphics', 'sci.med', 'soc.religion.christian']
Preprocessing texts...
Vectorizing with TF-IDF...
Training set: 3007 samples
Test set: 752 samples


In [0]:
# Part 2: Model Training and Evaluation

def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """Train a model and return evaluation metrics as a dictionary."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return {
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision (macro)': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'Recall (macro)': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'F1-score (macro)': f1_score(y_test, y_pred, average='macro', zero_division=0)
    }

# Train baseline models
baseline_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes': MultinomialNB(),
    'LinearSVC': LinearSVC(random_state=42)
}

baseline_results = []
for name, model in baseline_models.items():
    print(f"Training {name}...")
    baseline_results.append(evaluate_model(model, X_train, X_test, y_train, y_test, name))

baseline_df = pd.DataFrame(baseline_results)
display(baseline_df)

Training Logistic Regression...
Training Naive Bayes...
Training LinearSVC...


Model,Accuracy,Precision (macro),Recall (macro),F1-score (macro)
Logistic Regression,0.848404255319149,0.8466171227819185,0.8465126088770667,0.8447069976154217
Naive Bayes,0.8204787234042553,0.8383371835122233,0.8138856725603714,0.81189582453258
LinearSVC,0.839095744680851,0.8370072993260189,0.837002653471027,0.8350880214077716


In [0]:
# Task 1: Add Decision Tree and Random Forest models

def train_and_evaluate_tree_models(X_train, X_test, y_train, y_test):
    """Initialize, train, and evaluate Decision Tree and Random Forest models.
    Returns results in a consistent format matching baseline_results."""
    tree_models = {
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
    }

    results = []
    for name, model in tree_models.items():
        print(f"Training {name}...")
        results.append(evaluate_model(model, X_train, X_test, y_train, y_test, name))

    return pd.DataFrame(results)

tree_results_df = train_and_evaluate_tree_models(X_train, X_test, y_train, y_test)
display(tree_results_df)

Training Decision Tree...
Training Random Forest...


Model,Accuracy,Precision (macro),Recall (macro),F1-score (macro)
Decision Tree,0.6462765957446809,0.6387007712983522,0.6420021319042404,0.6377163260842261
Random Forest,0.7938829787234043,0.7907838803710752,0.7880171660668649,0.785111633294209


In [0]:
# Task 2: Combine and compare all model scores

def combine_scores(baseline_df, tree_df):
    """Combine scores from all models, sort by accuracy, and return a clean DataFrame."""
    combined = pd.concat([baseline_df, tree_df], ignore_index=True)
    combined = combined.sort_values('Accuracy', ascending=False).reset_index(drop=True)
    return combined

comparison_df = combine_scores(baseline_df, tree_results_df)
print("Model Comparison (sorted by accuracy):")
display(comparison_df)

Model Comparison (sorted by accuracy):


Model,Accuracy,Precision (macro),Recall (macro),F1-score (macro)
Logistic Regression,0.848404255319149,0.8466171227819185,0.8465126088770667,0.8447069976154217
LinearSVC,0.839095744680851,0.8370072993260189,0.837002653471027,0.8350880214077716
Naive Bayes,0.8204787234042553,0.8383371835122233,0.8138856725603714,0.81189582453258
Random Forest,0.7938829787234043,0.7907838803710752,0.7880171660668649,0.785111633294209
Decision Tree,0.6462765957446809,0.6387007712983522,0.6420021319042404,0.6377163260842261


In [0]:
# Task 3: Best model selection and testing

def select_best_model(comparison_df, X_train, y_train):
    """Select the best model based on F1-score, retrain on full training data, and return the model."""
    best_model_name = comparison_df.loc[comparison_df['F1-score (macro)'].idxmax(), 'Model']
    print(f"Best model by F1-score: {best_model_name}")

    model_map = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'Naive Bayes': MultinomialNB(),
        'LinearSVC': LinearSVC(random_state=42),
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
    }

    best_model = model_map[best_model_name]
    best_model.fit(X_train, y_train)
    return best_model, best_model_name

best_model, best_model_name = select_best_model(comparison_df, X_train, y_train)

# Test the best model on new texts
test_texts = [
    "The new graphics card offers incredible rendering speed and 4K resolution support.",
    "The patient showed symptoms of high fever and respiratory issues.",
    "The discussion about faith and belief systems continued in the church meeting."
]

# Preprocess and vectorize the new texts
processed_test = [preprocess_text(text) for text in test_texts]
X_new = vectorizer.transform(processed_test)
predictions = best_model.predict(X_new)

print(f"\nPredictions using best model ({best_model_name}):")
print("-" * 60)
for text, pred in zip(test_texts, predictions):
    category = label_names[pred]
    print(f"Text: {text}")
    print(f"Predicted category: {category}\n")

Best model by F1-score: Logistic Regression

Predictions using best model (Logistic Regression):
------------------------------------------------------------
Text: The new graphics card offers incredible rendering speed and 4K resolution support.
Predicted category: comp.graphics

Text: The patient showed symptoms of high fever and respiratory issues.
Predicted category: sci.med

Text: The discussion about faith and belief systems continued in the church meeting.
Predicted category: soc.religion.christian

